## Stacking

## Figure 6-11 — Aggregating predictions (прогноз на новом объекте)

![Figure 6-11. Aggregating predictions using a blending predictor](6-11.png)

**Смысл:** ансамбль на новом объекте. Каждый из 3 нижних предикторов даёт своё значение (3.1, 2.7, 2.9), финальный предиктор (**blender / meta learner**) берёт их как вход и делает итоговое предсказание (3.0). Это уже **обученная комбинация**, а не простое голосование.

## Figure 6-12 — Training the blender (обучение blender'а)

![Figure 6-12. Training the blender in a stacking ensemble](6-12.png)

**Смысл:** как собирают данные для обучения blender'а. Базовые модели предсказывают на held-out(отложенная , модель не училась на ней, а ток предсказывала) частях train (честные, out-of-fold предсказания... предсказания на том фолде , который модель не видела при обучении), эти числа становятся колонками `X_blender`, а `y_blender` — настоящие ответы. Затем blender обучается на этой таблице. После этого базовые модели переобучаются на всём train для будущих прогнозов.

---

## Схема обучения (упрощённо)

1. Обучаем базовые модели (например, Random Forest, SVM, Logistic Regression) на обычных признаках.
2. Для каждого объекта из train получаем **честное** предсказание от каждой базовой модели (через CV, как на 6-12).
3. Собираем таблицу для blender'а:
   - `X_blender = [предсказание RF, предсказание SVM, предсказание LR]`
   - `y_blender = настоящий ответ` (тот же `y` из train)
4. Обучаем blender на этой таблице. Он учится: «если RF=3.1, SVM=2.7, LR=2.9 → ответ 3.0».
5. Прогноз на новом объекте (как на 6-11): базовые модели дают 3 числа → blender → финальный ответ.

## ⚠️ Важная ловушка

Если брать предсказания **на тех же данных, где модели учились**, blender переобучится (он увидит «идеальные» ответы). Поэтому книга использует **`cross_val_predict`** — предсказания получаются «честными», на данных, которые модель в этот момент не видела.

---

## Кратко: две фазы

| Фаза | Рисунок | Что происходит |
|---|---|---|
| Обучение blender'а | 6-12 | Собираем честные предсказания → таблица → обучаем blender |
| Прогноз на новом объекте | 6-11 | Базовые модели → 3 числа → blender → финальный ответ |

 Figure 6-13 — Multilayer Stacking Ensemble

![Figure 6-13. Predictions in a multilayer stacking ensemble](6-13.png)

> «На самом деле можно обучить несколько разных blender'ов таким способом (например, один на линейной регрессии, другой на random forest regression), чтобы получить целый **слой** blender'ов, а затем добавить ещё один blender сверху, который и даст финальное предсказание, как показано на Figure 6-13. Вы можете выжать ещё пару капель производительности, но это будет стоить и времени обучения, и сложности системы.»

## Что показывает рисунок

Это **«stacking в несколько этажей»** (multilayer stacking):

- **Этаж 1 (низ):** базовые модели (RF, SVM, LR) → их предсказания.
- **Этаж 2 (средний):** несколько разных blender'ов (например, Blender A на линейной регрессии, Blender B на random forest) — каждый получает предсказания с этажа 1 и выдаёт своё предсказание.
- **Этаж 3 (верх):** финальный blender берёт предсказания с этажа 2 и делает итоговый ответ.

**Идея:** вместо одного blender'а — целый слой, который «смотрит» на базовые модели с разных сторон, и финальная модель объединяет их.

**Цена:** больше времени обучения + сложность системы. Выигрыш в accuracy обычно небольшой (книга приводит пример: stacking 92.8% vs soft voting 92%).

## Кратко

| Схема | Структура |
|---|---|
| Обычный stacking | базовые модели → 1 blender → ответ |
| Multilayer stacking | базовые модели → слой из нескольких blender'ов → финальный blender → ответ |

In [ ]:
from sklearn.datasets import make_moons
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

stacking_clf = StackingClassifier(
    estimators=[
        ("lr", LogisticRegression(random_state=42)),
        ("rf", RandomForestClassifier(random_state=42)),
        ("svc", SVC(probability=True, random_state=42)),
    ],
    final_estimator=RandomForestClassifier(random_state=43),
    cv=5,  # number of cross-validation folds
)
stacking_clf.fit(X_train, y_train)
print(stacking_clf.score(X_test, y_test)) # 0.928
# В начале главы получилось 0.92, при использовании VotingClassifier

0.928
